In [2]:
import os
from dotenv import load_dotenv, find_dotenv
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

load_dotenv(find_dotenv(), override=True)

MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT")
MINIO_ACCESS_KEY = os.getenv("MINIO_ACCESS_KEY")
MINIO_SECRET_KEY = os.getenv("MINIO_SECRET_KEY")

builder = SparkSession.builder \
    .appName("DMLDeltaLake") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT) \
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY) \
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY) \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .config("spark.hadoop.fs.s3a.threads.keepalivetime", "60000")

spark = configure_spark_with_delta_pip(
    builder,
    extra_packages=["org.apache.hadoop:hadoop-aws:3.4.2"]
).getOrCreate()

print("Spark inicializado!")

:: loading settings :: url = jar:file:/home/jeanvv/.cache/pypoetry/virtualenvs/apache-spark-minio-z9HCMJ8B-py3.11/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/jeanvv/.ivy2.5.2/cache
The jars for the packages stored in: /home/jeanvv/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-3d5a866f-709b-4403-895e-2712778214b8;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.1 in central
	found io.delta#delta-storage;4.4.1 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in local-m2-cache
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in local-m2-cache
	found com.google.code.findbugs#jsr305;3.0.2 in local-m2-cache
	found io.unitycatalog#un

Spark inicializado!


In [3]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col

caminho = "s3a://bronze/games"

df = spark.read.format("delta").load(caminho)
df.show(5, truncate=False)

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
26/10/14 18:30:35 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 8:>                                                          (0 + 1) / 1]

+---+---------------+-----------------+------------+---------+----------+-----------------------+----------------------------------------+---------------------+
|id |name           |platform         |release_date|metascore|user_score|developer              |publisher                               |genre                |
+---+---------------+-----------------+------------+---------+----------+-----------------------+----------------------------------------+---------------------+
|1  |#BLUD          |PC               |2024-06-18  |73       |5.6       |Exit 73 Studios        |['Humble Games']                        |Action Adventure     |
|2  |#DRIVE         |iOS (iPhone/iPad)|2021-02-16  |69       |6.8       |Dariusz Pietrala       |['PM Studios Inc.', 'Limited Run Games']|Auto Racing          |
|3  |#IDARB         |Xbox One         |2015-01-30  |77       |6.9       |Other Ocean Interactive|['Other Ocean Interactive']             |Party                |
|4  |#KILLALLZOMBIES|PC           

In [8]:
from pyspark.sql.types import (
    StructType, StructField, StringType,
    DateType, IntegerType, DoubleType
)
from datetime import date

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("platform", StringType(), True),
    StructField("release_date", DateType(), True),
    StructField("metascore", IntegerType(), True),
    StructField("user_score", DoubleType(), True),
    StructField("developer", StringType(), True),
    StructField("publisher", StringType(), True),
    StructField("genre", StringType(), True)
])

dados = [(
    None,
    "Delta Lake Demo Game",
    "PC",
    date(2026, 10, 7),
    85,
    8.5,
    "Demo Studio",
    "Demo Publisher",
    "Action"
)]

df_teste = spark.createDataFrame(dados, schema)

df_teste.write.format("delta").mode("append").save("s3a://bronze/games")

print("Registro inserido com sucesso!")

Registro inserido com sucesso!


In [9]:
spark.read.format("delta") \
    .load("s3a://bronze/games") \
    .filter("name = 'Delta Lake Demo Game'") \
    .show(truncate=False)

+----+--------------------+--------+------------+---------+----------+-----------+--------------+------+
|id  |name                |platform|release_date|metascore|user_score|developer  |publisher     |genre |
+----+--------------------+--------+------------+---------+----------+-----------+--------------+------+
|NULL|Delta Lake Demo Game|PC      |2026-10-07  |85       |8.5       |Demo Studio|Demo Publisher|Action|
+----+--------------------+--------+------------+---------+----------+-----------+--------------+------+



In [11]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col

caminho = "s3a://bronze/games"

delta_table = DeltaTable.forPath(spark, caminho)

delta_table.toDF() \
    .filter(col("name") == "Delta Lake Demo Game") \
    .show(truncate=False)

+----+--------------------+--------+------------+---------+----------+-----------+--------------+------+
|id  |name                |platform|release_date|metascore|user_score|developer  |publisher     |genre |
+----+--------------------+--------+------------+---------+----------+-----------+--------------+------+
|NULL|Delta Lake Demo Game|PC      |2026-10-07  |85       |8.5       |Demo Studio|Demo Publisher|Action|
+----+--------------------+--------+------------+---------+----------+-----------+--------------+------+



In [12]:
delta_table.update(
    condition=col("name") == "Delta Lake Demo Game",
    set={"metascore": col("metascore") * 0 + 90}
)

print("UPDATE executado!")

26/10/14 18:34:32 WARN UpdateCommand: Could not validate number of records due to missing statistics.
                                                                                

UPDATE executado!


In [13]:
delta_table.toDF() \
    .filter(col("name") == "Delta Lake Demo Game") \
    .select("name", "metascore") \
    .show(truncate=False)

+--------------------+---------+
|name                |metascore|
+--------------------+---------+
|Delta Lake Demo Game|90       |
+--------------------+---------+



In [14]:
delta_table.delete(
    condition=col("name") == "Delta Lake Demo Game"
)

print("DELETE executado!")

26/10/14 18:34:59 WARN DeleteCommand: Could not validate number of records due to missing statistics.
[Stage 65:=================================================>      (44 + 4) / 50]

DELETE executado!


In [15]:
delta_table.toDF() \
    .filter(col("name") == "Delta Lake Demo Game") \
    .show(truncate=False)

[Stage 71:======================================================> (49 + 1) / 50]

+---+----+--------+------------+---------+----------+---------+---------+-----+
|id |name|platform|release_date|metascore|user_score|developer|publisher|genre|
+---+----+--------+------------+---------+----------+---------+---------+-----+
+---+----+--------+------------+---------+----------+---------+---------+-----+



In [16]:
delta_table.history().select(
    "version",
    "timestamp",
    "operation",
    "operationMetrics"
).show(truncate=False)

+-------+-------------------+---------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp          |operation|operationMetrics                                                                                                                                                                                                                                                                                                              |
+-------+-------------------+---------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [17]:
df_final = spark.read.format("delta").load(caminho)

print(f"Total de registros na tabela: {df_final.count()}")
df_final.show(5, truncate=False)

Total de registros na tabela: 21910


[Stage 82:======================================================> (49 + 1) / 50]

+---+---------------+-----------------+------------+---------+----------+-----------------------+----------------------------------------+---------------------+
|id |name           |platform         |release_date|metascore|user_score|developer              |publisher                               |genre                |
+---+---------------+-----------------+------------+---------+----------+-----------------------+----------------------------------------+---------------------+
|1  |#BLUD          |PC               |2024-06-18  |73       |5.6       |Exit 73 Studios        |['Humble Games']                        |Action Adventure     |
|2  |#DRIVE         |iOS (iPhone/iPad)|2021-02-16  |69       |6.8       |Dariusz Pietrala       |['PM Studios Inc.', 'Limited Run Games']|Auto Racing          |
|3  |#IDARB         |Xbox One         |2015-01-30  |77       |6.9       |Other Ocean Interactive|['Other Ocean Interactive']             |Party                |
|4  |#KILLALLZOMBIES|PC           